## 1. Load data
We load the EV charging dataset and inspect the first rows.

In [32]:
import pandas as pd
import numpy as np

df = pd.read_csv("EVChargingStationUsage.csv", low_memory=False)

print("Shape:", df.shape)
display(df.head())


Shape: (259415, 33)


,Station Name,MAC Address,Org Name,Start Date,Start Time Zone,End Date,End Time Zone,Transaction Date (Pacific Time),Total Duration (hh:mm:ss),Charging Time (hh:mm:ss),...,Longitude,Currency,Fee,Ended By,Plug In Event Id,Driver Postal Code,User ID,County,System S/N,Model Number
0,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/29/2011 20:17,PDT,7/29/2011 23:20,PDT,7/29/2011 23:20,3:03:32,1:54:03,...,-122.160309,USD,0.0,Plug Out at Vehicle,3,95124.0,3284,NaN,NaN,NaN
1,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 0:00,PDT,7/30/2011 0:02,PDT,7/30/2011 0:02,0:02:06,0:01:54,...,-122.160309,USD,0.0,Customer,4,94301.0,4169,NaN,NaN,NaN
2,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 8:16,PDT,7/30/2011 12:34,PDT,7/30/2011 12:34,4:17:32,4:17:28,...,-122.160309,USD,0.0,Plug Out at Vehicle,5,94301.0,4169,NaN,NaN,NaN
3,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 14:51,PDT,7/30/2011 16:55,PDT,7/30/2011 16:55,2:03:24,2:02:58,...,-122.160309,USD,0.0,Customer,6,94302.0,2545,NaN,NaN,NaN
4,PALO ALTO CA / HAMILTON #1,000D:6F00:015A:9D76,City of Palo Alto,7/30/2011 18:51,PDT,7/30/2011 20:03,PDT,7/30/2011 20:03,1:11:24,0:43:54,...,-122.160309,USD,0.0,Plug Out at Vehicle,7,94043.0,3765,NaN,NaN,NaN


## 2. Basic quality checks
We check missing values, duplicates, and data types.

In [33]:
print("Missing values per column:")
display(df.isna().sum().sort_values(ascending=False))

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nData types:")
display(df.dtypes)

Missing values per column:


County                             84665
EVSE ID                            78948
System S/N                         78948
Model Number                       78948
Driver Postal Code                  8402
User ID                             7677
Currency                            1788
Ended By                             248
Transaction Date (Pacific Time)      209
Port Type                              9
Start Time Zone                        0
Postal Code                            0
Org Name                               0
Start Date                             0
Plug In Event Id                       0
Fee                                    0
Longitude                              0
Latitude                               0
Country                                0
City                                   0
State/Province                         0
End Date                               0
Address 1                              0
MAC Address                            0
Plug Type       


Exact duplicate rows: 4

Data types:


Station Name                           str
MAC Address                            str
Org Name                               str
Start Date                             str
Start Time Zone                        str
End Date                               str
End Time Zone                          str
Transaction Date (Pacific Time)        str
Total Duration (hh:mm:ss)              str
Charging Time (hh:mm:ss)               str
Energy (kWh)                       float64
GHG Savings (kg)                   float64
Gasoline Savings (gallons)         float64
Port Type                              str
Port Number                          int64
Plug Type                              str
EVSE ID                            float64
Address 1                              str
City                                   str
State/Province                         str
Postal Code                          int64
Country                                str
Latitude                           float64
Longitude  

## 3. Clean text columns
We remove extra spaces and normalize station names.

In [34]:
text_cols = ["Station Name", "Port Type", "Plug Type", "Ended By", "Start Time Zone", "End Time Zone"]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["Station Name"] = (
    df["Station Name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\s*#\s*", " #", regex=True)
)

df["Start Time Zone"] = df["Start Time Zone"].str.upper()
df["End Time Zone"] = df["End Time Zone"].str.upper()

## 4. Parse dates and durations
We convert date/time columns and duration strings into proper formats.

In [35]:
df["Start Date"] = pd.to_datetime(df["Start Date"], errors="coerce")
df["End Date"] = pd.to_datetime(df["End Date"], errors="coerce")
df["Transaction Date (Pacific Time)"] = pd.to_datetime(df["Transaction Date (Pacific Time)"], errors="coerce")

total_duration = pd.to_timedelta(df["Total Duration (hh:mm:ss)"], errors="coerce")
charging_duration = pd.to_timedelta(df["Charging Time (hh:mm:ss)"], errors="coerce")

## 5. Convert numeric fields
We convert important numeric columns to numeric values.

In [36]:
numeric_cols = ["Energy (kWh)", "Fee", "Latitude", "Longitude", "GHG Savings (kg)", "Gasoline Savings (gallons)"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## 6. Build cleaned session dataset
We remove exact duplicates and keep sessions with valid start/end timestamps.

In [37]:
invalid_time = (
    df["Start Date"].isna() |
    df["End Date"].isna() |
    (df["End Date"] <= df["Start Date"])
)

timestamp_duration_min = (df["End Date"] - df["Start Date"]).dt.total_seconds() / 60
total_duration_min = total_duration.dt.total_seconds() / 60
duration_diff_min = (timestamp_duration_min - total_duration_min).abs()

print("Invalid time rows:", invalid_time.sum())
print("Rows with duration mismatch > 2 min:", (duration_diff_min > 2).sum())
print("Negative energy rows:", (df["Energy (kWh)"] < 0).sum())

df = df.drop_duplicates().copy()

clean_mask = (
    df["Start Date"].notna() &
    df["End Date"].notna() &
    (df["End Date"] > df["Start Date"]) &
    df["Station Name"].notna()
)

df_clean = df.loc[clean_mask].copy()

print("\nRows after duplicate removal:", len(df))
print("Rows in cleaned dataset:", len(df_clean))

Invalid time rows: 83
Rows with duration mismatch > 2 min: 454
Negative energy rows: 0

Rows after duplicate removal: 259411
Rows in cleaned dataset: 259328


## 7. Convert timestamps to UTC
We convert start/end times to UTC usind their timezone labels.

In [38]:
zone_map = {"PST": "Etc/GMT+8", "PDT": "Etc/GMT+7", "UTC": "UTC"}

df_clean["Start_UTC"] = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns, UTC]")
df_clean["End_UTC"] = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns, UTC]")

for label, tz in zone_map.items():
    m1 = (df_clean["Start Time Zone"] == label) & df_clean["Start Date"].notna()
    df_clean.loc[m1, "Start_UTC"] = (
        df_clean.loc[m1, "Start Date"]
        .dt.tz_localize(tz)
        .dt.tz_convert("UTC")
    )

    m2 = (df_clean["End Time Zone"] == label) & df_clean["End Date"].notna()
    df_clean.loc[m2, "End_UTC"] = (
        df_clean.loc[m2, "End Date"]
        .dt.tz_localize(tz)
        .dt.tz_convert("UTC")
    )

df_clean["Elapsed_UTC_Minutes"] = (df_clean["End_UTC"] - df_clean["Start_UTC"]).dt.total_seconds() / 60

recorded_minutes = total_duration.loc[df_clean.index].dt.total_seconds() / 60
df_clean["UTC_Duration_Difference_Minutes"] = (df_clean["Elapsed_UTC_Minutes"] - recorded_minutes).abs()

display(
    df_clean[
        ["Start Date", "End Date", "Start_UTC", "End_UTC", "Elapsed_UTC_Minutes", "UTC_Duration_Difference_Minutes"]
    ].head()
)

,Start Date,End Date,Start_UTC,End_UTC,Elapsed_UTC_Minutes,UTC_Duration_Difference_Minutes
0,2011-07-29 20:17:00,2011-07-29 23:20:00,2011-07-30 03:17:00+00:00,2011-07-30 06:20:00+00:00,183.0,0.533333
1,2011-07-30 00:00:00,2011-07-30 00:02:00,2011-07-30 07:00:00+00:00,2011-07-30 07:02:00+00:00,2.0,0.100000
2,2011-07-30 08:16:00,2011-07-30 12:34:00,2011-07-30 15:16:00+00:00,2011-07-30 19:34:00+00:00,258.0,0.466667
3,2011-07-30 14:51:00,2011-07-30 16:55:00,2011-07-30 21:51:00+00:00,2011-07-30 23:55:00+00:00,124.0,0.600000
4,2011-07-30 18:51:00,2011-07-30 20:03:00,2011-07-31 01:51:00+00:00,2011-07-31 03:03:00+00:00,72.0,0.600000


## 8. Final time-quality filter
We keep only rows that are valid for time-based analysis.

In [39]:
time_quality_mask = (
    df_clean["Start_UTC"].notna() &
    df_clean["End_UTC"].notna() &
    (df_clean["Elapsed_UTC_Minutes"] > 0) &
    (
        df_clean["UTC_Duration_Difference_Minutes"].isna() |
        (df_clean["UTC_Duration_Difference_Minutes"] <= 2)
    )
)

df_time_clean = df_clean.loc[time_quality_mask].copy()
df_time_review = df_clean.loc[~time_quality_mask].copy()

print("Rows kept for analysis:", len(df_time_clean))
print("Rows sent to review:", len(df_time_review))

Rows kept for analysis: 259002
Rows sent to review: 326


## Data Quality Investigation

### 9. Data quality investigation (before clustering)
This section documents problematic records, gives examples, summarizes duration discrepancies, and states cleaning assumptions.

In [40]:
# 9.1 How many problematic records exist?
problem_flags = pd.DataFrame(index=df_clean.index)

problem_flags["missing_utc"] = df_clean["Start_UTC"].isna() | df_clean["End_UTC"].isna()
problem_flags["nonpositive_elapsed"] = df_clean["Elapsed_UTC_Minutes"] <= 0
problem_flags["large_duration_gap"] = df_clean["UTC_Duration_Difference_Minutes"] > 2
problem_flags["negative_energy"] = df_clean["Energy (kWh)"] < 0

problem_flags["any_problem"] = problem_flags.any(axis=1)

summary_counts = pd.Series({
    "total_rows_in_df_clean": len(df_clean),
    "rows_with_any_problem": int(problem_flags["any_problem"].sum()),
    "rows_missing_utc": int(problem_flags["missing_utc"].sum()),
    "rows_nonpositive_elapsed": int(problem_flags["nonpositive_elapsed"].sum()),
    "rows_duration_gap_gt_2min": int(problem_flags["large_duration_gap"].sum()),
    "rows_negative_energy": int(problem_flags["negative_energy"].sum()),
    "rows_kept_for_modeling_df_time_clean": len(df_time_clean),
    "rows_sent_to_review_df_time_review": len(df_time_review),
})

display(summary_counts.to_frame("count"))

,count
total_rows_in_df_clean,259328
rows_with_any_problem,326
rows_missing_utc,0
rows_nonpositive_elapsed,0
rows_duration_gap_gt_2min,326
rows_negative_energy,0
rows_kept_for_modeling_df_time_clean,259002
rows_sent_to_review_df_time_review,326


In [41]:
# 9.2 A few examples of problematic rows
example_cols = [
    "Station Name","User ID","Start Date","End Date",
    "Start Time Zone","End Time Zone",
    "Start_UTC","End_UTC","Elapsed_UTC_Minutes",
    "Total Duration (hh:mm:ss)","UTC_Duration_Difference_Minutes","Energy (kWh)"
]

examples = df_clean.loc[problem_flags["any_problem"], example_cols].copy()

print("Examples (first 15 problematic rows):")
display(examples.head(15))

Examples (first 15 problematic rows):


,Station Name,User ID,Start Date,End Date,Start Time Zone,End Time Zone,Start_UTC,End_UTC,Elapsed_UTC_Minutes,Total Duration (hh:mm:ss),UTC_Duration_Difference_Minutes,Energy (kWh)
77,PALO ALTO CA / HAMILTON #1,0,2011-08-18 11:59:00,2011-08-18 13:42:00,PDT,PDT,2011-08-18 18:59:00+00:00,2011-08-18 20:42:00+00:00,103.0,2:24:23,41.383333,5.014686
90,PALO ALTO CA / HAMILTON #2,2670,2011-08-20 14:43:00,2011-08-21 01:44:00,PDT,PDT,2011-08-20 21:43:00+00:00,2011-08-21 08:44:00+00:00,661.0,7:25:11,215.816667,5.268692
186,PALO ALTO CA / HIGH #4,0,2011-09-09 09:33:00,2011-09-09 10:38:00,PDT,PDT,2011-09-09 16:33:00+00:00,2011-09-09 17:38:00+00:00,65.0,1:09:46,4.766667,4.057364
275,PALO ALTO CA / HAMILTON #2,5618,2011-09-24 17:40:00,2011-09-24 18:41:00,PDT,PDT,2011-09-25 00:40:00+00:00,2011-09-25 01:41:00+00:00,61.0,11:33:33,632.550000,3.536726
2372,PALO ALTO CA / HAMILTON #1,29593,2012-03-26 18:23:00,2012-03-26 22:37:00,PDT,PDT,2012-03-27 01:23:00+00:00,2012-03-27 05:37:00+00:00,254.0,3:15:06,58.900000,7.010081
2389,PALO ALTO CA / HAMILTON #1,0,2012-03-27 14:40:00,2012-05-09 12:43:00,PDT,PDT,2012-03-27 21:40:00+00:00,2012-05-09 19:43:00+00:00,61803.0,22:59:59,60423.016667,0.476540
2431,PALO ALTO CA / BRYANT #1,16913,2012-03-30 08:07:00,2012-03-30 10:26:00,PDT,PDT,2012-03-30 15:07:00+00:00,2012-03-30 17:26:00+00:00,139.0,1:10:09,68.850000,4.234088
4611,PALO ALTO CA / HAMILTON #1,4447,2012-07-25 19:30:00,2012-07-27 01:46:00,PDT,PDT,2012-07-26 02:30:00+00:00,2012-07-27 08:46:00+00:00,1816.0,11:40:14,1115.766667,10.391167
5181,PALO ALTO CA / BRYANT #2,70103,2012-08-27 18:37:00,2012-08-30 12:30:00,PDT,PDT,2012-08-28 01:37:00+00:00,2012-08-30 19:30:00+00:00,3953.0,17:53:18,2879.700000,3.885966
5231,PALO ALTO CA / BRYANT #1,7157,2012-08-30 08:04:00,2012-08-30 08:47:00,PDT,PDT,2012-08-30 15:04:00+00:00,2012-08-30 15:47:00+00:00,43.0,0:26:06,16.900000,1.287699


In [42]:
# 9.3 Quick summary of duration discrepancies
duration_gap = df_clean["UTC_Duration_Difference_Minutes"]

print("Duration discrepancy summary (minutes):")
display(duration_gap.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\nTop 10 largest discrepancies:")
display(
    df_clean.sort_values("UTC_Duration_Difference_Minutes", ascending=False)[
        ["Station Name","Start Date","End Date","Total Duration (hh:mm:ss)",
         "Elapsed_UTC_Minutes","UTC_Duration_Difference_Minutes","Energy (kWh)"]
    ].head(10)
)

Duration discrepancy summary (minutes):


count    259328.000000
mean          6.242946
std         623.236018
min           0.000000
50%           0.300000
90%           0.683333
95%           0.783333
99%           0.916667
max      118437.850000
Name: UTC_Duration_Difference_Minutes, dtype: float64


Top 10 largest discrepancies:


,Station Name,Start Date,End Date,Total Duration (hh:mm:ss),Elapsed_UTC_Minutes,UTC_Duration_Difference_Minutes,Energy (kWh)
38666,PALO ALTO CA / MPL #6,2015-06-18 13:01:00,2015-09-08 20:24:00,1:25:09,118523.0,118437.850000,8.338037
38664,PALO ALTO CA / MPL #5,2015-06-18 12:10:00,2015-09-08 20:21:00,2:18:01,118571.0,118432.983333,13.213822
38606,PALO ALTO CA / MPL #4,2015-06-17 13:04:00,2015-08-27 09:57:00,0:30:13,102053.0,102022.783333,2.496953
38609,PALO ALTO CA / MPL #4,2015-06-17 14:24:00,2015-08-27 09:57:00,1:25:51,101973.0,101887.150000,8.336936
38644,PALO ALTO CA / MPL #4,2015-06-18 06:28:00,2015-08-27 09:57:00,1:25:22,101009.0,100923.633333,2.630938
38673,PALO ALTO CA / MPL #4,2015-06-18 13:55:00,2015-08-27 09:57:00,0:05:10,100562.0,100556.833333,0.226734
38674,PALO ALTO CA / MPL #4,2015-06-18 14:13:00,2015-08-27 09:57:00,0:15:18,100544.0,100528.700000,0.554137
39195,PALO ALTO CA / MPL #4,2015-06-30 13:14:00,2015-08-27 09:57:00,0:20:16,83323.0,83302.733333,1.184045
2389,PALO ALTO CA / HAMILTON #1,2012-03-27 14:40:00,2012-05-09 12:43:00,22:59:59,61803.0,60423.016667,0.476540
16244,PALO ALTO CA / HAMILTON #2,2013-10-30 17:28:00,2013-12-06 09:45:00,22:59:59,52877.0,51497.016667,1.738849


### Data-quality decisions (report text)

- We keep sessions only when UTC start/end timestamps are available, elapsed UTC duration is positive, and the UTC-vs-recorded duration mismatch is at most 2 minutes.
- We exclude records that fail these checks because they can distort hourly profiles and produce unreliable user behavior features for clustering.
- PST/PDT/UTC handling: we convert each timestamp using its own timezone label (PST, PDT, or UTC) and store a unified UTC timeline to make durations comparable across all records.
- Hourly energy allocation assumption: because only total session energy is known (not second-by-second charging power), we distribute session energy proportionally across the hours overlapped by the session duration.

## 10. Helper function: split one session into hours
We split each charging session into hourly overlaps.

In [43]:
def split_session_into_hours(start_utc, end_utc):
    rows = []
    h = start_utc.floor("h")

    while h < end_utc:
        h_end = h + pd.Timedelta(hours=1)
        overlap_start = max(start_utc, h)
        overlap_end = min(end_utc, h_end)

        overlap_hours = (overlap_end - overlap_start).total_seconds() / 3600
        if overlap_hours > 0:
            rows.append((h, overlap_hours))

        h = h_end

    return rows

## 11. Build hourly station and user tables
Energy is allocated to each hour proportionally to overlap duration.

In [44]:
# Build hourly station and user tables (updated naming + cleaner logic)
station_rows = []
user_rows = []

for _, r in df_time_clean.iterrows():
    parts = split_session_into_hours(r["Start_UTC"], r["End_UTC"])
    total_h = sum(x[1] for x in parts)

    energy = r["Energy (kWh)"] if pd.notna(r["Energy (kWh)"]) else 0.0

    for hour, occ_h in parts:
        e_h = (energy * occ_h / total_h) if total_h > 0 else 0.0

        station_rows.append({
            "Station Name": r["Station Name"],
            "Hour": hour,
            "Energy (kWh)": e_h,
            "Occupied_Port_Hours": occ_h,
            "Active_Session_Count": 1
        })

        if pd.notna(r["User ID"]) and str(r["User ID"]).strip() != "":
            user_rows.append({
                "User ID": str(r["User ID"]).strip(),
                "Hour": hour,
                "Energy (kWh)": e_h,
                "Occupied_Port_Hours": occ_h,
                "Active_Session_Count": 1
            })

hourly_energy = (
    pd.DataFrame(station_rows)
    .groupby(["Station Name", "Hour"], as_index=False)
    .sum()
)

hourly_user_usage = (
    pd.DataFrame(user_rows)
    .groupby(["User ID", "Hour"], as_index=False)
    .sum()
)

print("hourly_energy rows:", len(hourly_energy))
print("hourly_user_usage rows:", len(hourly_user_usage))
display(hourly_energy.head())

hourly_energy rows: 653425
hourly_user_usage rows: 861442


,Station Name,Hour,Energy (kWh),Occupied_Port_Hours,Active_Session_Count
0,PALO ALTO CA / BRYANT #1,2011-10-13 19:00:00+00:00,0.664281,0.183333,1
1,PALO ALTO CA / BRYANT #1,2011-10-13 20:00:00+00:00,0.966227,0.266667,1
2,PALO ALTO CA / BRYANT #1,2011-10-14 01:00:00+00:00,0.681748,0.183333,1
3,PALO ALTO CA / BRYANT #1,2011-10-14 02:00:00+00:00,3.718623,1.000000,1
4,PALO ALTO CA / BRYANT #1,2011-10-14 03:00:00+00:00,2.231174,0.600000,1


In [45]:
# Validation: energy conservation after hourly split
energy_before = df_time_clean["Energy (kWh)"].sum()
energy_after = hourly_energy["Energy (kWh)"].sum()
energy_diff = energy_after - energy_before

print("Energy before hourly split:", energy_before)
print("Energy after hourly split :", energy_after)
print("Difference:", energy_diff)
print("Conservation OK:", np.isclose(energy_before, energy_after, rtol=1e-10, atol=1e-6))

Energy before hourly split: 2212840.46164
Energy after hourly split : 2212840.46164
Difference: 0.0
Conservation OK: True


### Validation of hourly occupation allocation
In addition to energy conservation, the hourly transformation should preserve the total charger occupation time. The sum of all hourly overlap durations should therefore equal the total duration of the original cleaned charging sessions.

This check ensures that splitting charging sessions across hourly intervals does not create or remove charger occupation time.

In [46]:
# Validation: occupation-time conservation after hourly split

occupation_before = (
    (df_time_clean["End_UTC"] - df_time_clean["Start_UTC"])
    .dt.total_seconds()
    .sum()
    / 3600
)

occupation_after = hourly_energy["Occupied_Port_Hours"].sum()

occupation_diff = occupation_after - occupation_before

print("Occupation before hourly split:", occupation_before)
print("Occupation after hourly split :", occupation_after)
print("Difference:", occupation_diff)
print(
    "Conservation OK:",
    np.isclose(
        occupation_before,
        occupation_after,
        rtol=1e-10,
        atol=1e-6
    )
)

Occupation before hourly split: 643783.85
Occupation after hourly split : 643783.85
Difference: 0.0
Conservation OK: True


## 12. Add local-time features
We create local hour/day fields for behavior analysis.

In [47]:
hourly_energy["Hour_Local"] = hourly_energy["Hour"].dt.tz_convert("America/Los_Angeles")
hourly_energy["Hour_Of_Day"] = hourly_energy["Hour_Local"].dt.hour
hourly_energy["Weekday_Local"] = hourly_energy["Hour_Local"].dt.day_of_week

hourly_user_usage["Hour_Local"] = hourly_user_usage["Hour"].dt.tz_convert("America/Los_Angeles")
hourly_user_usage["Hour_Of_Day"] = hourly_user_usage["Hour_Local"].dt.hour
hourly_user_usage["Weekday_Local"] = hourly_user_usage["Hour_Local"].dt.day_of_week

display(hourly_energy.head())
display(hourly_user_usage.head())

,Station Name,Hour,Energy (kWh),Occupied_Port_Hours,Active_Session_Count,Hour_Local,Hour_Of_Day,Weekday_Local
0,PALO ALTO CA / BRYANT #1,2011-10-13 19:00:00+00:00,0.664281,0.183333,1,2011-10-13 12:00:00-07:00,12,3
1,PALO ALTO CA / BRYANT #1,2011-10-13 20:00:00+00:00,0.966227,0.266667,1,2011-10-13 13:00:00-07:00,13,3
2,PALO ALTO CA / BRYANT #1,2011-10-14 01:00:00+00:00,0.681748,0.183333,1,2011-10-13 18:00:00-07:00,18,3
3,PALO ALTO CA / BRYANT #1,2011-10-14 02:00:00+00:00,3.718623,1.000000,1,2011-10-13 19:00:00-07:00,19,3
4,PALO ALTO CA / BRYANT #1,2011-10-14 03:00:00+00:00,2.231174,0.600000,1,2011-10-13 20:00:00-07:00,20,3


,User ID,Hour,Energy (kWh),Occupied_Port_Hours,Active_Session_Count,Hour_Local,Hour_Of_Day,Weekday_Local
0,0,2011-08-02 19:00:00+00:00,1.959811,0.566667,1,2011-08-02 12:00:00-07:00,12,1
1,0,2011-08-02 20:00:00+00:00,1.556320,0.450000,1,2011-08-02 13:00:00-07:00,13,1
2,0,2011-08-05 03:00:00+00:00,2.307848,0.783333,1,2011-08-04 20:00:00-07:00,20,3
3,0,2011-08-05 04:00:00+00:00,2.946189,1.000000,1,2011-08-04 21:00:00-07:00,21,3
4,0,2011-08-05 05:00:00+00:00,2.946189,1.000000,1,2011-08-04 22:00:00-07:00,22,3


## Pre-clustering checks
We verify user coverage, missing values, filter low-activity users, and build a user-level feature table.

1) Check number of users


In [48]:
n_users = hourly_user_usage["User ID"].nunique()
print("Unique users:", n_users)

display(
    hourly_user_usage["User ID"]
    .value_counts()
    .head(10)
    .rename("hourly_rows_per_user")
)

Unique users: 21434


User ID
458893    5917
128279    4638
278457    4206
2670      4205
100243    3981
196691    3657
4447      3651
143839    3410
485553    3301
453469    3274
Name: hourly_rows_per_user, dtype: int64

2) Check missing values in key columns

In [49]:
key_cols = ["User ID", "Hour", "Energy (kWh)", "Occupied_Port_Hours"]
print("Missing values in key columns:")
display(hourly_user_usage[key_cols].isna().sum())

# keep only valid key rows
hourly_user_usage = hourly_user_usage.dropna(subset=key_cols).copy()
print("Rows after key-column NA drop:", len(hourly_user_usage))

Missing values in key columns:


User ID                0
Hour                   0
Energy (kWh)           0
Occupied_Port_Hours    0
dtype: int64

Rows after key-column NA drop: 861442


3) Filter very low-activity users

#### User activity threshold
An initial threshold of 24 active hourly observations was considered to ensure that each user had sufficient data to construct a meaningful charging profile. However, the threshold analysis showed that this would retain only 3,855 users, approximately 18% of the identifiable user population.

Since the project description does not specify a minimum number of observations required for user clustering, a lower threshold of 10 active hourly observations was selected. This provides a better balance between excluding extremely sparse user profiles and retaining a sufficiently large and representative user population for clustering.

The threshold refers to active hourly observations rather than charging sessions, since a single charging session may span several hourly intervals. The choice of threshold is therefore a methodological decision rather than a requirement of the project.

In [50]:
# Pre-clustering: justify user-activity threshold
user_activity = (
    hourly_user_usage.groupby("User ID")
    .size()
    .rename("active_hour_rows")
    .reset_index()
)

print("Activity distribution:")
display(user_activity["active_hour_rows"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

thresholds = [5, 10, 24, 50, 100]
total_users = user_activity["User ID"].nunique()

rows = []
for t in thresholds:
    kept = (user_activity["active_hour_rows"] >= t).sum()
    rows.append({
        "threshold": t,
        "users_kept": int(kept),
        "pct_users_kept": round(100 * kept / total_users, 2)
    })

threshold_table = pd.DataFrame(rows)
display(threshold_table)

# choose threshold
min_rows = 10
active_users = user_activity[user_activity["active_hour_rows"] >= min_rows]["User ID"]

hourly_user_usage_f = hourly_user_usage[
    hourly_user_usage["User ID"].isin(active_users)
].copy()

print("Chosen threshold:", min_rows)
print("Users before filter:", hourly_user_usage["User ID"].nunique())
print("Users after filter:", hourly_user_usage_f["User ID"].nunique())
print("Rows after filter:", len(hourly_user_usage_f))

Activity distribution:


count    21434.000000
mean        40.190445
std        178.528918
min          1.000000
50%          5.000000
75%         15.000000
90%         58.000000
95%        150.350000
99%        780.000000
max       5917.000000
Name: active_hour_rows, dtype: float64

,threshold,users_kept,pct_users_kept
0,5,11448,53.41
1,10,7106,33.15
2,24,3855,17.99
3,50,2397,11.18
4,100,1457,6.80


Chosen threshold: 10
Users before filter: 21434
Users after filter: 7106
Rows after filter: 806668


4) Buid user-feature table (one row = one user)

In [51]:
# total behavior features
base_features = (
    hourly_user_usage_f.groupby("User ID")
    .agg(
        total_energy_kwh=("Energy (kWh)", "sum"),
        avg_energy_per_active_hour=("Energy (kWh)", "mean"),
        total_occupied_hours=("Occupied_Port_Hours", "sum"),
        avg_occupied_per_active_hour=("Occupied_Port_Hours", "mean"),
        active_hour_rows=("Hour", "count")
    )
)

# hour-of-day behavior profile (0..23)
hour_profile = (
    hourly_user_usage_f.pivot_table(
        index="User ID",
        columns="Hour_Of_Day",
        values="Energy (kWh)",
        aggfunc="sum",
        fill_value=0
    )
)

# normalize to shares (so we cluster behavior pattern, not only volume)
hour_profile = hour_profile.div(hour_profile.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
hour_profile.columns = [f"share_h{int(c):02d}" for c in hour_profile.columns]

# weekday/weekend preference
tmp = hourly_user_usage_f.copy()
tmp["is_weekend"] = tmp["Weekday_Local"].isin([5, 6])

week_profile = (
    tmp.groupby(["User ID", "is_weekend"])["Energy (kWh)"]
    .sum()
    .unstack(fill_value=0)
)

week_profile.columns = ["weekday_energy", "weekend_energy"] if False in week_profile.columns and True in week_profile.columns else week_profile.columns
if "weekday_energy" not in week_profile.columns:
    week_profile["weekday_energy"] = 0.0
if "weekend_energy" not in week_profile.columns:
    week_profile["weekend_energy"] = 0.0

week_total = (week_profile["weekday_energy"] + week_profile["weekend_energy"]).replace(0, np.nan)
week_profile["weekday_share"] = (week_profile["weekday_energy"] / week_total).fillna(0)
week_profile["weekend_share"] = (week_profile["weekend_energy"] / week_total).fillna(0)

# final clustering table
user_features = base_features.join(hour_profile, how="left").join(
    week_profile[["weekday_share", "weekend_share"]], how="left"
).fillna(0)

print("user_features shape:", user_features.shape)
display(
    user_features
    .sort_values("User ID", ascending=True)
    .head(10)
)

user_features shape: (7106, 31)


,total_energy_kwh,avg_energy_per_active_hour,total_occupied_hours,avg_occupied_per_active_hour,active_hour_rows,share_h00,share_h01,share_h02,share_h03,share_h04,...,share_h16,share_h17,share_h18,share_h19,share_h20,share_h21,share_h22,share_h23,weekday_share,weekend_share
User ID,,,,,,,,,,,,,,,,,,,,,
0,3074.856252,2.005777,1231.666667,0.803436,1533,0.015124,0.012955,0.011828,0.010646,0.010100,...,0.045648,0.056241,0.068904,0.093393,0.081121,0.059328,0.033894,0.021426,0.804281,0.195719
100061V,34.569000,2.659154,9.333333,0.717949,13,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
100161V,33.824000,3.074909,7.150000,0.650000,11,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
100243,5322.069112,1.336867,3382.533333,0.849669,3981,0.045902,0.044648,0.044387,0.044587,0.044509,...,0.046774,0.070262,0.078238,0.075302,0.070671,0.060282,0.050690,0.047624,0.747973,0.252027
100293,96.191890,1.479875,48.433333,0.745128,65,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.007889,0.071549,0.103232,0.091180,0.075932,0.036787,0.000000,0.386570,0.613430
100393,11.870527,0.741908,7.366667,0.460417,16,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.047445,0.309203,0.173475,0.254918,0.166811,0.000000,0.000000,0.000000,0.869801,0.130199
100523,315.665840,2.270977,87.616667,0.630336,139,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.062158,0.037910,0.045397,0.029924,0.029188,0.012109,0.003941,0.000000,0.919019,0.080981
100673,40.299187,3.663562,8.233333,0.748485,11,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.086153,0.104694,0.104694,0.104694,0.104694,0.104694,0.022684,0.560111,0.439889
1007181,18.097000,1.508083,7.100000,0.591667,12,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.131477,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.540697,0.459303


### Investigation of User ID = 0
During the user-level data quality assessment, User ID = 0 was identified as a potentially special identifier. This was investigated separately to determine whether it represented missing user information.

The dataset contains missing User ID values independently from records where User ID = 0, indicating that zero is not simply the result of missing values being replaced during preprocessing. Sessions associated with User ID = 0 also occur across multiple charging stations.

However, the available data does not provide sufficient evidence to determine whether User ID = 0 represents a legitimate user or a special identifier used by the source system. Therefore, these records are retained rather than removed based on an unsupported assumption. Records with missing User ID values are excluded from user-level clustering because they cannot be attributed to a specific user.

In [52]:
# Investigate User ID = 0 separately from missing user IDs

user_id_clean = df_time_clean["User ID"].astype(str).str.strip()
user_0 = df_time_clean[user_id_clean == "0"]

print("User ID = 0 sessions:", len(user_0))
print("Missing User ID sessions:", df_time_clean["User ID"].isna().sum())
print("Stations used by User ID = 0:", user_0["Station Name"].nunique())
print("First session:", user_0["Start Date"].min())
print("Last session:", user_0["Start Date"].max())

display(
    user_0["Station Name"]
    .value_counts()
    .rename("Sessions")
    .to_frame()
)

User ID = 0 sessions: 450
Missing User ID sessions: 7646
Stations used by User ID = 0: 14
First session: 2011-08-02 12:26:00
Last session: 2015-09-02 20:16:00


,Sessions
Station Name,
PALO ALTO CA / HAMILTON #1,95
PALO ALTO CA / BRYANT #2,90
PALO ALTO CA / BRYANT #1,89
PALO ALTO CA / HAMILTON #2,87
PALO ALTO CA / HIGH #4,62
PALO ALTO CA / RINCONADA LIB 3,7
PALO ALTO CA / RINCONADA LIB 2,4
PALO ALTO CA / MPL #5,3
PALO ALTO CA / MPL #6,3
